# 04 — Google Chirp 3 API validation pilot

Test Google Cloud Speech-to-Text V2 **`chirp_3`** on the exact same five validation clips as the completed Whisper API pilot. Compare corpus WER/CER and transcript pairs using unchanged references and normalization. **Five Google requests, one automatic-language condition, 50.976 seconds of original audio.** The saved Whisper results are embedded; no OpenAI calls are repeated. This is a compatibility pilot, not the final CMI benchmark.

Use a **fresh Google Colab CPU session**. Install dependencies, restart the session, then resume at **1.2**. The configured Project ID is **`research-asr`**, with the Speech-to-Text API and billing enabled as reported by the researcher. Sign in to the Google account that has access to that project. A permanent API key or downloaded service-account key is unnecessary for this notebook. Google sign-in credentials remain only in the runtime and are excluded from exports.

Section **3.2** uploads audio and incurs Google Cloud usage. At the [standard V2 planning rate](https://cloud.google.com/speech-to-text/pricing?authuser=0) of $0.016/minute, rounding each clip up to a second gives 54 seconds, approximately **$0.0144** for this five-request pilot, before taxes and any account-specific pricing. This is an estimate, not a spending cap or verified invoice.

The [Chirp 3 guide](https://docs.cloud.google.com/speech-to-text/docs/models/chirp-3) documents `language_codes=["auto"]` and the `us` multi-region. Nepali (`ne-NP`) is listed as Preview. Automatic language handling must be checked empirically for these mixed clips. Do not silently switch languages or regions to improve the measured scores. No reference words, CMI labels, prompts or previous predictions are sent to Google.

Requests are sequential and use no automatic transcription retries. Each attempt is checkpointed before submission and every returned body before scoring. Rerunning skips received responses; failed or uncertain attempts stop for investigation. Download the final ZIP even after an error. A local checkpoint does not survive deletion of the Colab runtime unless you download and restore it.

**Latest October 23 recovery update:** your first run returned three responses, timed out on clip 4, and never attempted clip 5. The explicit retry of clip 4 returned **HTTP 503** after 184.33 seconds. Section **2.4** now restores that latest saved checkpoint, including both failed attempts. After signing in with **3.1**, run **3.4** to submit only the unattempted clip 5. Clip 4 is preserved as a service failure and is not retried again. The first three responses are reused unchanged. Client read timeout remains 300 seconds. Both original ZIPs remain preserved as research evidence; this still cannot produce a full five-clip score while clip 4 is missing.


### 1.1 Install Dependencies, Then Restart

After this cell, choose **Runtime → Restart session**, then resume at **1.2**. Do not rerun installation in the restarted session. These dataset/audio versions match the completed pilot. Conflicts with unused preinstalled Colab applications may be reported.


In [ ]:
%pip install -q "requests==2.32.4" "google-auth==2.49.0" "datasets==4.4.1" "huggingface-hub==0.36.2" "numpy==2.1.3" "soundfile==0.13.1" "jiwer==4.0.0"


### 1.2 Import Libraries

No GPU, local model, or cloud transcription is needed for setup.


In [ ]:
import base64
import hashlib
import io
import json
import math
import platform
import re
import sys
import time
import unicodedata
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import google.auth
from google.auth.transport.requests import Request
import huggingface_hub
import numpy as np
import requests
import soundfile as sf
from datasets import load_dataset
from IPython.display import Audio, display
from jiwer import cer, wer

assert version('huggingface-hub') == huggingface_hub.__version__ == '0.36.2', 'Install, restart, then resume at 1.2.'
package_versions = {name: version(name) for name in ['requests', 'google-auth', 'datasets', 'huggingface-hub', 'numpy', 'soundfile', 'jiwer']}
print('Versions:', package_versions)


### 1.3 Load the Saved Whisper API Baseline

The manifest and automatic-language predictions are embedded from your completed export. No ZIP upload or additional Whisper request is needed.


In [ ]:
baseline = json.loads('{"created_at_utc": "2026-10-22T10:56:23.043636+00:00", "api_model": "whisper-1", "dataset": {"id": "saileshbro/nepali-cs-asr", "revision": "dd00ca41dd38774cd6097c372c16939d7f3746df", "config": "default", "split": "validation"}, "manifest": [{"transcription": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_devanagari": "होइन धेरै जसो स्टुडेन्ट के हुन्छन् तपाईँको सटुडय एकदम हारड गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_romanized": "hoina dherai jaso student ke hunchan tapaiko study ekdam hard garchan matlab dina rata 1 garchan uniharu", "cmi": 18.75, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "JF_HlxEXcHQ", "start_ms": 90896, "end_ms": 99488, "duration_s": 8.592, "clip_id": "validation/JF_HlxEXcHQ/90896-99488", "decoded_duration_s": 8.592, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "5d7728fbbec30b6f4f42110ecd48a76d19dea181673460552219a1f9beefcaa8"}, {"transcription": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "transcription_devanagari": "यसलाई ईसिलय यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त सो हाफ लाइफ को meaning के हो आधा हुने हो नि त सो तिम्रो लाइक", "transcription_romanized": "yaslai easily yatikai pani nikalna milne thiyo 25% bhaneko ho ni ta so half life ko meaning ke ho aadha hune ho ni ta so timro like", "cmi": 26.92, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 633264, "end_ms": 642192, "duration_s": 8.928, "clip_id": "validation/zJYS-XNDP2Q/633264-642192", "decoded_duration_s": 8.928, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "a19babdc7287647007c9675646769c74d8dd186b780fb5ce9ae8a74f887c52ef"}, {"transcription": "यो हामीले निकाल्ने हो नि त mass भनेको basically N N हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "transcription_devanagari": "यो हामीले निकाल्ने हो नि त म्यास भनेको बेसिकली एन एन हाम्रो कति निक्लियो यति निक्लियो इन्टु molar म्यास molar म्यास हाम्रो कति हुन्छ भन्दाखेरि", "transcription_romanized": "yo hamile nikalne ho ni ta mass bhaneko basically N N hamro kati nikliyo yati nikliyo into molar mass molar mass hamro kati huncha bhandakheri", "cmi": 36.0, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 1512240, "end_ms": 1525520, "duration_s": 13.28, "clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "decoded_duration_s": 13.28, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "574ae80ba5da885329f71a80a96040083e9b54e656a9f3f7f9aa15b8980c94b4"}, {"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9"}, {"transcription": "Electric charge field potential capacitor DC circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "transcription_devanagari": "इलेक्ट्रिक चार्ज फिल्ड पोटेनशल क्यापासिटर डीसी सर्किट च्याप्टर हरू छन् होइन जसमध्ये मोसटलय तपाईँको कन्सेप्ट", "transcription_romanized": "Electric charge field potential capacitor DC circuit chapter haru chan hoina jasamadhye mostly tapaiko concept", "cmi": 33.33, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1169296, "end_ms": 1178352, "duration_s": 9.056, "clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "decoded_duration_s": 9.056, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "52e59651d4a7af08a78092d441cebc8414550ca2721ceb5476644d8d1d8162a1"}], "summary": {"auto": {"successful_clips": 5, "expected_clips": 5, "complete": true, "wer": 0.7297297297297297, "cer": 0.4876190476190476}, "forced_ne": {"successful_clips": 5, "expected_clips": 5, "complete": true, "wer": 0.7297297297297297, "cer": 0.4876190476190476}}, "run_spec": {"dataset": {"id": "saileshbro/nepali-cs-asr", "revision": "dd00ca41dd38774cd6097c372c16939d7f3746df", "config": "default", "split": "validation"}, "manifest": [{"transcription": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_devanagari": "होइन धेरै जसो स्टुडेन्ट के हुन्छन् तपाईँको सटुडय एकदम हारड गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_romanized": "hoina dherai jaso student ke hunchan tapaiko study ekdam hard garchan matlab dina rata 1 garchan uniharu", "cmi": 18.75, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "JF_HlxEXcHQ", "start_ms": 90896, "end_ms": 99488, "duration_s": 8.592, "clip_id": "validation/JF_HlxEXcHQ/90896-99488", "decoded_duration_s": 8.592, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "5d7728fbbec30b6f4f42110ecd48a76d19dea181673460552219a1f9beefcaa8"}, {"transcription": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "transcription_devanagari": "यसलाई ईसिलय यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त सो हाफ लाइफ को meaning के हो आधा हुने हो नि त सो तिम्रो लाइक", "transcription_romanized": "yaslai easily yatikai pani nikalna milne thiyo 25% bhaneko ho ni ta so half life ko meaning ke ho aadha hune ho ni ta so timro like", "cmi": 26.92, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 633264, "end_ms": 642192, "duration_s": 8.928, "clip_id": "validation/zJYS-XNDP2Q/633264-642192", "decoded_duration_s": 8.928, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "a19babdc7287647007c9675646769c74d8dd186b780fb5ce9ae8a74f887c52ef"}, {"transcription": "यो हामीले निकाल्ने हो नि त mass भनेको basically N N हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "transcription_devanagari": "यो हामीले निकाल्ने हो नि त म्यास भनेको बेसिकली एन एन हाम्रो कति निक्लियो यति निक्लियो इन्टु molar म्यास molar म्यास हाम्रो कति हुन्छ भन्दाखेरि", "transcription_romanized": "yo hamile nikalne ho ni ta mass bhaneko basically N N hamro kati nikliyo yati nikliyo into molar mass molar mass hamro kati huncha bhandakheri", "cmi": 36.0, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 1512240, "end_ms": 1525520, "duration_s": 13.28, "clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "decoded_duration_s": 13.28, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "574ae80ba5da885329f71a80a96040083e9b54e656a9f3f7f9aa15b8980c94b4"}, {"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9"}, {"transcription": "Electric charge field potential capacitor DC circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "transcription_devanagari": "इलेक्ट्रिक चार्ज फिल्ड पोटेनशल क्यापासिटर डीसी सर्किट च्याप्टर हरू छन् होइन जसमध्ये मोसटलय तपाईँको कन्सेप्ट", "transcription_romanized": "Electric charge field potential capacitor DC circuit chapter haru chan hoina jasamadhye mostly tapaiko concept", "cmi": 33.33, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1169296, "end_ms": 1178352, "duration_s": 9.056, "clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "decoded_duration_s": 9.056, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "52e59651d4a7af08a78092d441cebc8414550ca2721ceb5476644d8d1d8162a1"}], "request_settings": {"model": "whisper-1", "response_format": "verbose_json", "temperature": 0.0}, "language_modes": [["auto", null], ["forced_ne", "ne"]], "payload_sha256": {"validation/JF_HlxEXcHQ/90896-99488": "05010983d2ef654342f94082f5ad6c2df58f7529c8612c4369dd4a80099e9daf", "validation/zJYS-XNDP2Q/633264-642192": "347deefc571d7c56d5d1f1b2dd7f1792f32b0ec4f78687853e24f1e878a109fa", "validation/zJYS-XNDP2Q/1512240-1525520": "ee29c9847d2acc9b91594e58e1a264b73c0acf1dc091da6aa2153dcf5af6cd61", "validation/TN8IzjVNnzs/1205232-1216352": "d4c9ba8ec46ce1550380582fb124363a1bf9ddbaa510e1533f04a9e2fef9030b", "validation/TN8IzjVNnzs/1169296-1178352": "3a99696fcb29b26be75bbbd85e9d185cd50a14dd32e0a0cbd40dba65891cb194"}, "endpoint": "https://api.openai.com/v1/audio/transcriptions", "max_retries": 0}, "source_archive": "whisper_api_pilot_20261022T105623043636Z.zip", "source_sha256": "8733d683d43667eca01cb519a796574a7c9d3064fe7a4f8229e52714ff66ef8c", "predictions": [{"clip_id": "validation/JF_HlxEXcHQ/90896-99488", "mode": "auto", "hypothesis": "आइन्या, धेरी जस्सो स्ट्रिडेंट के हुन्छन तोपाईको स्ट्रेडी एकदाम हार्ड गर्छन, मतलब दिन रात एक गर्छन हुनियारु?", "normalized_reference": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "normalized_hypothesis": "आइन्या धेरी जस्सो स्ट्रिडेंट के हुन्छन तोपाईको स्ट्रेडी एकदाम हार्ड गर्छन मतलब दिन रात एक गर्छन हुनियारु", "wer": 0.7647058823529411, "cer": 0.5789473684210527}, {"clip_id": "validation/zJYS-XNDP2Q/633264-642192", "mode": "auto", "hypothesis": "25% 25% 25% 25% 25%", "normalized_reference": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25 भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "normalized_hypothesis": "25 25 25 25 25", "wer": 0.9629629629629629, "cer": 0.9789473684210527}, {"clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "mode": "auto", "hypothesis": "यो हमने निकाल नियों ता मास मानिगो बिसिक्लि  यन आमरो कति निकलियो एति निकलियो इन्टू मॉलर मास मॉलर मास आमरो कति हुँँछा", "normalized_reference": "यो हामीले निकाल्ने हो नि त mass भनेको basically n n हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "normalized_hypothesis": "यो हमने निकाल नियों ता मास मानिगो बिसिक्लि यन आमरो कति निकलियो एति निकलियो इन्टू मॉलर मास मॉलर मास आमरो कति हुँँछा", "wer": 0.88, "cer": 0.5964912280701754}, {"clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "mode": "auto", "hypothesis": "Physical Interpretation इदि गरना सक्यो भने इस बाड़ा हमी Question from the Predict Table, last year को Question बाड़ा गरना सक्छो Marks लगना सजीलो बैदिंजा Repetition Rate", "normalized_reference": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "normalized_hypothesis": "physical interpretation इदि गरना सक्यो भने इस बाड़ा हमी question from the predict table last year को question बाड़ा गरना सक्छो marks लगना सजीलो बैदिंजा repetition rate", "wer": 0.5185185185185185, "cer": 0.22666666666666666}, {"clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "mode": "auto", "hypothesis": "electric charge, field potential, capacitor, DC circuit chapter हरू सहन अहिन जस मदि मस्टेलि तपाइंको concept", "normalized_reference": "electric charge field potential capacitor dc circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "normalized_hypothesis": "electric charge field potential capacitor dc circuit chapter हरू सहन अहिन जस मदि मस्टेलि तपाइंको concept", "wer": 0.4, "cer": 0.18888888888888888}]}')

PROJECT_ID = 'research-asr'
REGION = 'us'
DATASET_ID = baseline['dataset']['id']
DATASET_REVISION = baseline['dataset']['revision']
SPLIT = baseline['dataset']['split']
expected_clips = {row['clip_id']: row for row in baseline['manifest']}
assert SPLIT == 'validation' and len(expected_clips) == len(baseline['manifest']) == 5
assert re.fullmatch(r'[a-z][a-z0-9-]{4,28}[a-z0-9]', PROJECT_ID), 'Use the project ID, not its display name.'
assert REGION == 'us', 'This pilot freezes the us multi-region.'
print('Google project:', PROJECT_ID, '| region:', REGION)
print('Saved Whisper baseline WER:', f"{baseline['summary']['auto']['wer']:.2%}")


### 2.1 Audio Validation and Normalization

Reuse the waveform validation, identity hashes, and mixed-script scoring rules from the previous notebooks.


In [ ]:
columns = ['audio', 'transcription', 'transcription_devanagari', 'transcription_romanized', 'cmi', 'speech_type', 'speaker_id', 'video_id', 'start_ms', 'end_ms', 'duration_s']

def normalize_text(text):
    text = unicodedata.normalize('NFC', text).lower()
    text = text.translate(str.maketrans('०१२३४५६७८९', '0123456789'))
    text = ''.join(' ' if unicodedata.category(char).startswith('P') else char for char in text)
    return ' '.join(text.split())

assert normalize_text('Hello,  SCHOOL!') == 'hello school'
assert normalize_text('नेपालमा १२ school छन्।') == 'नेपालमा 12 school छन्'
assert normalize_text('cafe\u0301') == normalize_text('café')
assert normalize_text('school') != normalize_text('स्कुल')
assert normalize_text(' ' + chr(9) + chr(10)) == ''
assert wer(['a b', 'c d'], ['a', 'c d e']) == 0.5
assert wer('a b', '') == 1.0
assert wer('a', 'a b c') == 2.0
assert cer('ab', 'a') == 0.5
print('Normalization and scoring self-checks passed.')

def prepare_clip(row):
    video_id = row.get('video_id')
    start, end = row.get('start_ms'), row.get('end_ms')
    if (
        not isinstance(video_id, str) or not video_id.strip()
        or any(not isinstance(value, int) or isinstance(value, bool) for value in (start, end))
        or not 0 <= start < end
    ):
        raise ValueError('Missing or invalid video_id/start_ms/end_ms; inspect clip provenance.')
    key = f'{SPLIT}/{video_id}/{start}-{end}'
    reference, cmi = row.get('transcription'), row.get('cmi')
    if not isinstance(reference, str) or not normalize_text(reference):
        raise ValueError(f'Empty or invalid reference for {key}.')
    if isinstance(cmi, bool) or not isinstance(cmi, (int, float)) or not math.isfinite(cmi) or not 0 <= cmi <= 50:
        raise ValueError(f'Invalid bilingual CMI for {key}: {cmi}.')
    encoded = row.get('audio')
    if not isinstance(encoded, dict) or not isinstance(encoded.get('bytes'), bytes) or not encoded['bytes']:
        raise ValueError(f'Expected embedded WAV bytes for {key}; inspect the dataset format.')
    waveform, sample_rate = sf.read(io.BytesIO(encoded['bytes']), dtype='float32', always_2d=True)
    if sample_rate != 16000 or waveform.shape[0] == 0 or waveform.shape[1] == 0 or not np.isfinite(waveform).all():
        raise ValueError(f'Invalid or non-finite audio for {key}; expected nonempty 16 kHz audio.')
    channels = waveform.shape[1]
    audio = waveform.mean(axis=1)
    duration = len(audio) / sample_rate
    expected = row.get('duration_s')
    if (
        not 0 < duration <= 30
        or isinstance(expected, bool) or not isinstance(expected, (int, float)) or not math.isfinite(expected)
        or abs(duration - expected) > 0.02
        or abs(duration - (end - start) / 1000) > 0.02
    ):
        raise ValueError(f'Audio duration does not match metadata/provenance for {key}.')
    identity = {name: row[name] for name in columns if name != 'audio'}
    identity.update(
        clip_id=key, decoded_duration_s=duration, sample_rate=sample_rate, source_channels=channels,
        audio_sha256=hashlib.sha256(audio.tobytes()).hexdigest(),
    )
    return {'identity': identity, 'audio': audio}


def inspect_clip(clip):
    identity, audio = clip['identity'], clip['audio']
    print('CLIP:', identity['clip_id'])
    print('REFERENCE:', identity['transcription'])
    print('CMI:', identity['cmi'], '| SPEECH TYPE:', identity['speech_type'])
    print('Waveform shape:', audio.shape, '| dtype:', audio.dtype, '| type:', type(audio))
    print('Source channels:', identity['source_channels'], '| Sample rate:', identity['sample_rate'])
    print('Dataset duration:', identity['duration_s'], '| Decoded duration:', identity['decoded_duration_s'])
    print('Min:', float(audio.min()), '| Max:', float(audio.max()))
    print('Mean absolute amplitude:', float(np.abs(audio).mean()))
    display(Audio(audio, rate=identity['sample_rate']))


### 2.2 Recover and Verify the Same Five Audio Clips

Read the pinned validation split and check all five waveform hashes and metadata against the saved manifest. Retain the original WAV bytes. This scan may take several minutes; it makes no paid transcription calls.


In [ ]:
stream = load_dataset(DATASET_ID, name='default', split=SPLIT, revision=DATASET_REVISION, streaming=True)
stream = stream.decode(False).select_columns(columns)
found = {}
for row in stream:
    key = f'{SPLIT}/{row["video_id"]}/{row["start_ms"]}-{row["end_ms"]}'
    if key not in expected_clips:
        continue
    if key in found:
        raise ValueError(f'Ambiguous duplicate provenance: {key}.')
    clip = prepare_clip(row)
    expected = expected_clips[key]
    if clip['identity'] != expected:
        raise ValueError(f'Audio hash or metadata changed for {key}; do not compare different inputs.')
    clip['upload_bytes'] = row['audio']['bytes']
    found[key] = clip
missing = set(expected_clips) - set(found)
if missing:
    raise ValueError(f'Missing original clips: {sorted(missing)}.')
clips = [found[row['clip_id']] for row in baseline['manifest']]
print('Verified five exact audio hashes and all recorded clip metadata.')


### 2.3 Prepare the Google Request and Checkpoint

Use the [V2 Recognize REST method](https://docs.cloud.google.com/speech-to-text/docs/reference/rest/v2/projects.locations.recognizers/recognize), the `us` endpoint, implicit recognizer `_`, `model="chirp_3"`, automatic WAV decoding and `languageCodes=["auto"]`. Each clip is shorter than the synchronous one-minute limit. The request contains only recognition configuration and base64-encoded original WAV bytes.

The checkpoint contains public project/settings, audio hashes, raw recognition response bodies, response metadata and safe failure information. It excludes audio, request headers and Google credentials. Received bodies are saved before parsing. Empty results count as empty transcripts; malformed responses are retained for investigation. Failed/uncertain requests are never automatically resent.


In [ ]:
API_MODEL = 'chirp_3'
ENDPOINT = f'https://{REGION}-speech.googleapis.com/v2/projects/{PROJECT_ID}/locations/{REGION}/recognizers/_:recognize'
REQUEST_CONFIG = {'autoDecodingConfig': {}, 'languageCodes': ['auto'], 'model': API_MODEL}
CHECKPOINT_PATH = Path('chirp3_api_pilot_checkpoint.json')
run_spec = {
    'dataset': baseline['dataset'], 'manifest': [clip['identity'] for clip in clips],
    'project_id': PROJECT_ID, 'region': REGION, 'endpoint': ENDPOINT,
    'request_config': REQUEST_CONFIG, 'mode': 'auto', 'max_retries': 0,
    'payload_sha256': {clip['identity']['clip_id']: hashlib.sha256(clip['upload_bytes']).hexdigest() for clip in clips},
}


def save_checkpoint(state):
    temporary = CHECKPOINT_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(state, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(CHECKPOINT_PATH)


def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'run_spec': run_spec, 'predictions': []}
    state = json.loads(CHECKPOINT_PATH.read_text(encoding='utf-8'))
    if state['run_spec'] != run_spec:
        raise ValueError('Checkpoint inputs/settings differ. Preserve it separately; do not overwrite it.')
    jobs = [row['clip_id'] for row in state['predictions']]
    if len(set(jobs)) != len(jobs) or not set(jobs) <= set(expected_clips):
        raise ValueError('Checkpoint has duplicate or unknown clips.')
    history = state.get('previous_attempts', [])
    if len({row['clip_id'] for row in history}) != len(history):
        raise ValueError('More than one historical retry for a clip is unsupported.')
    if any(row['status'] != 'api_error' or row.get('error_type') != 'ReadTimeout' for row in history):
        raise ValueError('Only explicitly reviewed read timeouts may enter retry history.')
    for row in state['predictions'] + history:
        if row['clip_id'] not in expected_clips:
            raise ValueError('Unknown historical clip.')
        if row['status'] not in {'received', 'api_error', 'pending'} or row['mode'] != 'auto':
            raise ValueError('Unknown checkpoint status/mode.')
        if any(row[key] != value for key, value in expected_clips[row['clip_id']].items()):
            raise ValueError('Checkpoint reference/audio identity differs.')
    return state


def response_text(body):
    if not isinstance(body, dict) or 'error' in body:
        raise ValueError('Unexpected recognition response. Export the raw checkpoint for investigation.')
    results = body.get('results', [])
    if not isinstance(results, list):
        raise ValueError('Response results must be a list.')
    parts = []
    for result in results:
        if not isinstance(result, dict):
            raise ValueError('Recognition result must be an object.')
        alternatives = result.get('alternatives')
        if not isinstance(alternatives, list) or not alternatives or not isinstance(alternatives[0], dict):
            raise ValueError('Recognition result has no valid top alternative.')
        text = alternatives[0].get('transcript')
        if not isinstance(text, str):
            raise ValueError('Top transcript must be a string.')
        parts.append(text.strip())
    return ' '.join(parts).strip()


def scored_predictions(state):
    rows = []
    for record in state['predictions']:
        row = dict(record)
        if row['status'] == 'received':
            try:
                hypothesis = response_text(row.get('response'))
            except ValueError as error:
                row['parse_error'] = str(error)
            else:
                ref, hyp = normalize_text(row['transcription']), normalize_text(hypothesis)
                if not ref:
                    raise ValueError('Empty normalized reference cannot be scored.')
                row.update(hypothesis=hypothesis, normalized_reference=ref, normalized_hypothesis=hyp,
                           wer=wer(ref, hyp), cer=cer(ref.replace(' ', ''), hyp.replace(' ', '')))
        rows.append(row)
    return rows


def summarize(rows):
    scored = [row for row in rows if 'wer' in row]
    complete = len(scored) == len(expected_clips) and {row['clip_id'] for row in scored} == set(expected_clips)
    result = {'successful_clips': len(scored), 'expected_clips': len(expected_clips), 'complete': complete}
    if complete:
        refs, hyps = [r['normalized_reference'] for r in scored], [r['normalized_hypothesis'] for r in scored]
        result.update(wer=wer(refs, hyps), cer=cer([r.replace(' ', '') for r in refs], [h.replace(' ', '') for h in hyps]),
                      request_latency_s=sum(r['request_latency_s'] for r in scored))
    return result


def run_api_pilot(skip_saved_service_errors=False):
    state = load_checkpoint()
    completed = {row['clip_id']: row for row in state['predictions']}
    for clip in clips:
        identity = clip['identity']
        if identity['clip_id'] in completed:
            prior = completed[identity['clip_id']]
            if prior['status'] != 'received':
                if skip_saved_service_errors and prior['status'] == 'api_error' and prior.get('http_status') == 503:
                    print('Preserved saved HTTP-503 failure without resending:', identity['clip_id'])
                    continue
                print('STOP: failed/uncertain request. Export the checkpoint and investigate before any retry.')
                return state
            if not isinstance(prior.get('response'), dict):
                print('STOP: unexpected saved response. Export the checkpoint for investigation.')
                return state
            response_text(prior['response'])
            print('Already saved:', identity['clip_id'])
            continue
        payload = {'config': REQUEST_CONFIG, 'content': base64.b64encode(clip['upload_bytes']).decode('ascii')}
        headers = {'Content-Type': 'application/json', 'x-goog-user-project': PROJECT_ID}
        # Refresh OAuth before starting a billable attempt; never save headers or credentials.
        credentials.before_request(auth_request, 'POST', ENDPOINT, headers)
        record = {**identity, 'mode': 'auto', 'status': 'pending', 'requested_at_utc': datetime.now(timezone.utc).isoformat(),
                  'request_timeout_s': {'connect': 15, 'read': 300}}
        state['predictions'].append(record)
        save_checkpoint(state)
        started = time.perf_counter()
        try:
            response = session.post(ENDPOINT, headers=headers, json=payload, timeout=(15, 300), allow_redirects=False)
        except requests.RequestException as error:
            record.update(status='api_error', error_type=type(error).__name__, request_latency_s=time.perf_counter() - started)
            save_checkpoint(state)
            print('STOP:', record['error_type'], '| completion/billing may be uncertain; no retry. Export partial results.')
            return state
        record.update(http_status=response.status_code, request_latency_s=time.perf_counter() - started)
        if not 200 <= response.status_code < 300:
            # Keep safe status fields; do not serialize server error messages or exception objects.
            record.update(status='api_error', error_type='HTTPError')
            save_checkpoint(state)
            print('STOP: HTTP', response.status_code, '| 400: configuration; 401/403: account, permissions, API or billing; 429: quota.')
            print('No automatic retry. Export partial results before investigating.')
            return state
        record.update(status='received', response_body=response.text)
        save_checkpoint(state)
        # Preserve even non-JSON paid responses before parsing.
        try:
            body = json.loads(record['response_body'])
        except ValueError:
            print('STOP: non-JSON response saved. Export partial results.')
            return state
        record.update(response=body)
        del record['response_body']
        save_checkpoint(state)
        try:
            text = response_text(body)
        except ValueError:
            print('STOP: unexpected response shape saved. Export partial results.')
            return state
        completed[identity['clip_id']] = record
        print('Saved:', identity['clip_id'])
        print('CHIRP 3:', text)
    return state


def retry_single_read_timeout():
    state = load_checkpoint()
    failures = [row for row in state['predictions'] if row['status'] != 'received']
    if not failures:
        print('No failed request to retry. Use 3.2 to resume any unattempted clips.')
        return state
    if len(failures) != 1 or failures[0]['status'] != 'api_error' or failures[0].get('error_type') != 'ReadTimeout':
        raise ValueError('Recovery allows exactly one ReadTimeout, not pending requests or other API errors. Export and investigate.')
    failed = failures[0]
    if any(row['clip_id'] == failed['clip_id'] for row in state.get('previous_attempts', [])):
        raise ValueError('This clip already used its one explicit retry. Export and investigate; do not repeat charges.')
    state.setdefault('previous_attempts', []).append({**failed, 'retry_requested_at_utc': datetime.now(timezone.utc).isoformat()})
    state['predictions'] = [row for row in state['predictions'] if row['clip_id'] != failed['clip_id']]
    save_checkpoint(state)
    print('One explicit retry authorized by this recovery-cell execution; earlier failed attempt retained.')
    return run_api_pilot()


checkpoint = load_checkpoint()
print('Previously attempted clips:', len(checkpoint['predictions']), '/ 5')
print('No transcription requests made by this cell.')


### 2.4 Restore Your Saved October 23 Partial Run

This cell embeds `checkpoint.json` from your latest returned `chirp3_api_pilot_20261023T090553705776Z.zip`: three unchanged responses, the clip-4 HTTP-503 failure, and its earlier read timeout in retry history. It makes no cloud calls. It restores only when there is no local checkpoint; existing progress is never overwritten. The run settings and original audio identities must match. Proceed to Google sign-in in 3.1, then use **3.4** to attempt only clip 5. Running 3.2 first safely stops at the saved error.


In [ ]:
saved_checkpoint = json.loads('{"created_at_utc": "2026-10-23T07:59:25.516407+00:00", "run_spec": {"dataset": {"id": "saileshbro/nepali-cs-asr", "revision": "dd00ca41dd38774cd6097c372c16939d7f3746df", "config": "default", "split": "validation"}, "manifest": [{"transcription": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_devanagari": "होइन धेरै जसो स्टुडेन्ट के हुन्छन् तपाईँको सटुडय एकदम हारड गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_romanized": "hoina dherai jaso student ke hunchan tapaiko study ekdam hard garchan matlab dina rata 1 garchan uniharu", "cmi": 18.75, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "JF_HlxEXcHQ", "start_ms": 90896, "end_ms": 99488, "duration_s": 8.592, "clip_id": "validation/JF_HlxEXcHQ/90896-99488", "decoded_duration_s": 8.592, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "5d7728fbbec30b6f4f42110ecd48a76d19dea181673460552219a1f9beefcaa8"}, {"transcription": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "transcription_devanagari": "यसलाई ईसिलय यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त सो हाफ लाइफ को meaning के हो आधा हुने हो नि त सो तिम्रो लाइक", "transcription_romanized": "yaslai easily yatikai pani nikalna milne thiyo 25% bhaneko ho ni ta so half life ko meaning ke ho aadha hune ho ni ta so timro like", "cmi": 26.92, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 633264, "end_ms": 642192, "duration_s": 8.928, "clip_id": "validation/zJYS-XNDP2Q/633264-642192", "decoded_duration_s": 8.928, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "a19babdc7287647007c9675646769c74d8dd186b780fb5ce9ae8a74f887c52ef"}, {"transcription": "यो हामीले निकाल्ने हो नि त mass भनेको basically N N हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "transcription_devanagari": "यो हामीले निकाल्ने हो नि त म्यास भनेको बेसिकली एन एन हाम्रो कति निक्लियो यति निक्लियो इन्टु molar म्यास molar म्यास हाम्रो कति हुन्छ भन्दाखेरि", "transcription_romanized": "yo hamile nikalne ho ni ta mass bhaneko basically N N hamro kati nikliyo yati nikliyo into molar mass molar mass hamro kati huncha bhandakheri", "cmi": 36.0, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 1512240, "end_ms": 1525520, "duration_s": 13.28, "clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "decoded_duration_s": 13.28, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "574ae80ba5da885329f71a80a96040083e9b54e656a9f3f7f9aa15b8980c94b4"}, {"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9"}, {"transcription": "Electric charge field potential capacitor DC circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "transcription_devanagari": "इलेक्ट्रिक चार्ज फिल्ड पोटेनशल क्यापासिटर डीसी सर्किट च्याप्टर हरू छन् होइन जसमध्ये मोसटलय तपाईँको कन्सेप्ट", "transcription_romanized": "Electric charge field potential capacitor DC circuit chapter haru chan hoina jasamadhye mostly tapaiko concept", "cmi": 33.33, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1169296, "end_ms": 1178352, "duration_s": 9.056, "clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "decoded_duration_s": 9.056, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "52e59651d4a7af08a78092d441cebc8414550ca2721ceb5476644d8d1d8162a1"}], "project_id": "research-asr", "region": "us", "endpoint": "https://us-speech.googleapis.com/v2/projects/research-asr/locations/us/recognizers/_:recognize", "request_config": {"autoDecodingConfig": {}, "languageCodes": ["auto"], "model": "chirp_3"}, "mode": "auto", "max_retries": 0, "payload_sha256": {"validation/JF_HlxEXcHQ/90896-99488": "05010983d2ef654342f94082f5ad6c2df58f7529c8612c4369dd4a80099e9daf", "validation/zJYS-XNDP2Q/633264-642192": "347deefc571d7c56d5d1f1b2dd7f1792f32b0ec4f78687853e24f1e878a109fa", "validation/zJYS-XNDP2Q/1512240-1525520": "ee29c9847d2acc9b91594e58e1a264b73c0acf1dc091da6aa2153dcf5af6cd61", "validation/TN8IzjVNnzs/1205232-1216352": "d4c9ba8ec46ce1550380582fb124363a1bf9ddbaa510e1533f04a9e2fef9030b", "validation/TN8IzjVNnzs/1169296-1178352": "3a99696fcb29b26be75bbbd85e9d185cd50a14dd32e0a0cbd40dba65891cb194"}}, "predictions": [{"transcription": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_devanagari": "होइन धेरै जसो स्टुडेन्ट के हुन्छन् तपाईँको सटुडय एकदम हारड गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_romanized": "hoina dherai jaso student ke hunchan tapaiko study ekdam hard garchan matlab dina rata 1 garchan uniharu", "cmi": 18.75, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "JF_HlxEXcHQ", "start_ms": 90896, "end_ms": 99488, "duration_s": 8.592, "clip_id": "validation/JF_HlxEXcHQ/90896-99488", "decoded_duration_s": 8.592, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "5d7728fbbec30b6f4f42110ecd48a76d19dea181673460552219a1f9beefcaa8", "mode": "auto", "status": "received", "requested_at_utc": "2026-10-23T07:59:25.614919+00:00", "http_status": 200, "request_latency_s": 1.0910805359999358, "response": {"metadata": {"totalBilledDuration": "9s", "requestId": "7001424c-0000-278c-bc1a-d4f547ed6f78", "prompt": "Transcribe the following speech segment. Follow these specific instructions for formatting the answer:\\n* Only output the transcript, with no newlines.\\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 100 instead of one hundred etc.\\n* Only output the transcript, exactly as said in the speech segment.\\n* Do not continue the speech segment.\\n* Include everything said in the speech segment.\\n* Use mathematical notation where needed, eg. x ^ 6Y.\\n* If there is background speech that is not part of the main conversation, discard the background speech and only output the main conversation.\\n* If there is no clear speech, just output [BACKGROUND].\\n* Respond in the original language of the speech."}, "results": [{"alternatives": [{"transcript": "हैन, धेरै जसो स्टुडेन्ट के हुन्छन्? तपाईंको स्टडी एकदम हार्ड गर्छन्। मतलब दिन-रात एक गर्छन् उनीहरू।"}], "languageCode": "ne"}]}}, {"transcription": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "transcription_devanagari": "यसलाई ईसिलय यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त सो हाफ लाइफ को meaning के हो आधा हुने हो नि त सो तिम्रो लाइक", "transcription_romanized": "yaslai easily yatikai pani nikalna milne thiyo 25% bhaneko ho ni ta so half life ko meaning ke ho aadha hune ho ni ta so timro like", "cmi": 26.92, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 633264, "end_ms": 642192, "duration_s": 8.928, "clip_id": "validation/zJYS-XNDP2Q/633264-642192", "decoded_duration_s": 8.928, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "a19babdc7287647007c9675646769c74d8dd186b780fb5ce9ae8a74f887c52ef", "mode": "auto", "status": "received", "requested_at_utc": "2026-10-23T07:59:26.715093+00:00", "http_status": 200, "request_latency_s": 0.5607896230003462, "response": {"metadata": {"totalBilledDuration": "9s", "requestId": "7a27f213-0000-2763-be84-d4f547ecc56c", "prompt": "Transcribe the following speech segment. Follow these specific instructions for formatting the answer:\\n* Only output the transcript, with no newlines.\\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 100 instead of one hundred etc.\\n* Only output the transcript, exactly as said in the speech segment.\\n* Do not continue the speech segment.\\n* Include everything said in the speech segment.\\n* Use mathematical notation where needed, eg. x ^ 6Y.\\n* If there is background speech that is not part of the main conversation, discard the background speech and only output the main conversation.\\n* If there is no clear speech, just output [BACKGROUND].\\n* Respond in the original language of the speech."}, "results": [{"alternatives": [{"transcript": "easy 25%"}], "languageCode": "en"}]}}, {"transcription": "यो हामीले निकाल्ने हो नि त mass भनेको basically N N हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "transcription_devanagari": "यो हामीले निकाल्ने हो नि त म्यास भनेको बेसिकली एन एन हाम्रो कति निक्लियो यति निक्लियो इन्टु molar म्यास molar म्यास हाम्रो कति हुन्छ भन्दाखेरि", "transcription_romanized": "yo hamile nikalne ho ni ta mass bhaneko basically N N hamro kati nikliyo yati nikliyo into molar mass molar mass hamro kati huncha bhandakheri", "cmi": 36.0, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 1512240, "end_ms": 1525520, "duration_s": 13.28, "clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "decoded_duration_s": 13.28, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "574ae80ba5da885329f71a80a96040083e9b54e656a9f3f7f9aa15b8980c94b4", "mode": "auto", "status": "received", "requested_at_utc": "2026-10-23T07:59:27.279146+00:00", "http_status": 200, "request_latency_s": 18.697143709000102, "response": {"metadata": {"totalBilledDuration": "14s", "requestId": "7f6d16df-0000-231e-9449-d4f547ff3004", "prompt": "Transcribe the following speech segment. Follow these specific instructions for formatting the answer:\\n* Only output the transcript, with no newlines.\\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 100 instead of one hundred etc.\\n* Only output the transcript, exactly as said in the speech segment.\\n* Do not continue the speech segment.\\n* Include everything said in the speech segment.\\n* Use mathematical notation where needed, eg. x ^ 6Y.\\n* If there is background speech that is not part of the main conversation, discard the background speech and only output the main conversation.\\n* If there is no clear speech, just output [BACKGROUND].\\n* Respond in the original language of the speech."}, "results": [{"alternatives": [{"transcript": "यह हमने निकाल लिया होता। मास, 100, 100, 100, 100, 100,"}], "languageCode": "hi"}]}}, {"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9", "mode": "auto", "status": "api_error", "requested_at_utc": "2026-10-23T09:02:17.425225+00:00", "request_timeout_s": {"connect": 15, "read": 300}, "http_status": 503, "request_latency_s": 184.33043807700005, "error_type": "HTTPError"}], "previous_attempts": [{"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9", "mode": "auto", "status": "api_error", "requested_at_utc": "2026-10-23T07:59:45.979667+00:00", "error_type": "ReadTimeout", "request_latency_s": 120.10326540999995, "retry_requested_at_utc": "2026-10-23T09:02:17.312360+00:00"}]}')

assert saved_checkpoint['run_spec'] == run_spec, 'Saved recovery settings differ; do not reuse this checkpoint.'
if CHECKPOINT_PATH.exists():
    checkpoint = load_checkpoint()
    print('Kept existing local checkpoint; no overwrite.')
else:
    save_checkpoint(saved_checkpoint)
    checkpoint = load_checkpoint()
    print('Restored latest October 23 checkpoint: three responses, HTTP 503, and earlier timeout history.')
print('Received:', sum(row['status'] == 'received' for row in checkpoint['predictions']), '/ 5')


### 3.1 Sign In to Your Google Cloud Account

Use the account that owns or can use project `research-asr`. Follow the Colab sign-in dialog. No downloaded JSON key is needed; the service account created earlier can remain unused. This cell prepares authenticated access but does not submit audio.

If transcription later returns HTTP 403, confirm the selected project ID, API enablement, billing, and account permissions. Recognition requires `speech.recognizers.recognize` (included in Speech Client); charging the quota project also requires `serviceusage.services.use`. Project owners usually already have these permissions. Do not paste access tokens into code or chat.


In [ ]:
from google.colab import auth

auth.authenticate_user(project_id=PROJECT_ID)
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'], quota_project_id=PROJECT_ID)
auth_request = Request()
session = requests.Session()
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=0))
print('Google sign-in ready for project:', PROJECT_ID)
print('API access will be checked by the five-clip pilot; no audio submitted yet.')


### 3.2 Run Five Chirp 3 Transcriptions

This cell sends the five original audio clips to Google and incurs Cloud usage. It does not rerun Whisper. Complete this pilot before considering the larger benchmark. If an error occurs, run **4.2** to download partial results; do not delete the checkpoint and blindly retry.


In [ ]:
checkpoint = run_api_pilot()
print('Received responses:', sum(row['status'] == 'received' for row in checkpoint['predictions']), '/ 5')


### 3.3 Earlier Read-Timeout Recovery — Skip for the Latest Export

The one explicit retry of clip 4 has already been used and returned HTTP 503. With the latest checkpoint, this cell makes no request and directs you to **3.4**. Do not delete the failure history to retry again.

For a historical checkpoint containing only one read timeout, running this cell explicitly authorizes at most one retry, which may incur another charge even if the original request was processed. It preserves the earlier attempt and never retries based on transcription accuracy. Pending requests and other errors are not eligible.


In [ ]:
checkpoint = load_checkpoint()
if any(row.get('http_status') == 503 for row in checkpoint['predictions']):
    print('Read-timeout retry is unavailable for the saved HTTP-503 error. Proceed to 3.4; do not retry clip 4.')
else:
    checkpoint = retry_single_read_timeout()
print('Received responses:', sum(row['status'] == 'received' for row in checkpoint['predictions']), '/ 5')


### 3.4 Submit Only the Unattempted Clip 5

Run after restoring the latest checkpoint (2.4) and Google sign-in (3.1). This explicitly continues past a **saved HTTP-503 service failure without resending it**. It reuses clips 1–3 and submits the previously unattempted clip 5, once. Clip 4 and its earlier timeout remain in the export. Other failure types and pending requests still stop the run; no transcription is automatically retried.

This new clip-5 request incurs Google usage. Rerunning the cell skips any received clip-5 response. If it fails, export the results and investigate; do not delete the checkpoint. The result will still have incomplete five-clip coverage because clip 4 has no transcript.


In [ ]:
checkpoint = run_api_pilot(skip_saved_service_errors=True)
print('Received responses:', sum(row['status'] == 'received' for row in checkpoint['predictions']), '/ 5')


### 4.1 Compare Chirp 3 with the Saved Whisper API Results

Display corpus WER/CER only with all five valid responses. An empty transcript counts as deletions; request/response failures stay separate from ASR errors. Join the top transcript from each sequential recognition result. Preserve every returned alternative, language code, request ID and billed duration in the raw response.

Request latency includes network and service time. Five validation clips cannot establish a CMI relationship or a general winner. Inspect whether English is retained, transliterated, omitted or misrecognized before defining the final audit.


In [ ]:
checkpoint = load_checkpoint()
predictions = scored_predictions(checkpoint)
summary = summarize(predictions)
print('Whisper API automatic:', baseline['summary']['auto'])
print('Chirp 3 automatic:', summary)
old_predictions = {row['clip_id']: row for row in baseline['predictions']}
for row in predictions:
    print('\nCLIP:', row['clip_id'], '| CMI:', row['cmi'])
    print('REFERENCE:', row['transcription'])
    print('WHISPER API:', old_predictions[row['clip_id']]['hypothesis'])
    print('CHIRP 3:', row.get('hypothesis', '[No scorable response]'))
    if 'wer' in row:
        assert row['normalized_reference'] == old_predictions[row['clip_id']]['normalized_reference']
        print('Chirp WER:', f"{row['wer']:.2%}", '| CER:', f"{row['cer']:.2%}")
        print('Returned metadata:', row['response'].get('metadata', {}))
    else:
        print('Status:', row['status'], '| parse error:', row.get('parse_error'), '| HTTP:', row.get('http_status'))


### 4.2 Download Results and Checkpoint

Run even after an API error or skipped summary. The ZIP preserves partial results and settings for diagnosis. It contains JSON only; no audio, base64 request payloads, OAuth tokens, headers or account credentials. Download and send back this ZIP for independent score verification. Preserve it before replacing the runtime.


In [ ]:
checkpoint = load_checkpoint()
predictions = scored_predictions(checkpoint)
created_at = datetime.now(timezone.utc)
bundle = {
    'created_at_utc': created_at.isoformat(), 'purpose': 'five-clip Chirp 3 versus saved Whisper API validation pilot',
    'api_model': API_MODEL, 'run_spec': run_spec, 'dataset': baseline['dataset'],
    'manifest': [clip['identity'] for clip in clips], 'whisper_api_baseline': baseline,
    'predictions': predictions, 'summary': {'auto': summarize(predictions)},
    'previous_attempts': checkpoint.get('previous_attempts', []),
    'limitations': ['Five validation clips; no final CMI finding or broad model ranking.',
                    'Service internals and exact hosted checkpoint revision unavailable; Nepali support is Preview.',
                    'API request latency includes network/service time; billing must be reconciled separately.'],
    'runtime': {'python': sys.version, 'platform': platform.platform(), 'packages': package_versions},
}
archive_path = Path('chirp3_api_pilot_' + created_at.strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
with zipfile.ZipFile(archive_path, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('results.json', json.dumps(bundle, ensure_ascii=False, indent=2, allow_nan=False))
    archive.writestr('checkpoint.json', json.dumps(checkpoint, ensure_ascii=False, indent=2, allow_nan=False))
print('Saved:', archive_path.resolve())
from google.colab import files
files.download(str(archive_path))
